# Medical Insurance Cost Analysis
### Predicting Individual Medical Costs Using Demographic and Lifestyle Features

## 1. Project Overview
This notebook analyses a medical insurance dataset containing 1,338 records of policyholders. We investigate how age, sex, BMI, number of children, smoking status, and region influence annual medical charges, and use visualisations and statistical tests to quantify each factor's impact.

## 2. Learning Objectives
- Explore insurance cost distributions and identify skewness
- Quantify the effect of smoking on medical charges
- Analyse correlations between BMI, age, and charges
- Perform hypothesis testing (t-tests, ANOVA) on categorical groups
- Build informative multi-panel visualisations

## 3. Business / Research Problem
**Question:** Which demographic and lifestyle factors most strongly predict medical insurance costs? How much more do smokers pay compared to non-smokers after controlling for age and BMI?

## 4. Why This Analysis Matters
Insurance companies need actuarial insights to set fair premiums. Understanding cost drivers helps policyholders make informed health decisions and helps providers design targeted wellness programmes that reduce claims.

## 5. Dataset Overview
The dataset contains the following columns:
- `age` — policyholder age (18–64)
- `sex` — male / female
- `bmi` — body mass index
- `children` — number of dependents covered
- `smoker` — yes / no
- `region` — US region (northeast, southeast, southwest, northwest)
- `charges` — annual medical insurance cost (target)

## 6. Dataset Source and License Notes
- **Kaggle:** `mirichoi0218/insurance`
- **Origin:** Public domain dataset commonly used in ML education
- **License:** Open / CC0

## 7. Environment Setup

In [1]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','scipy']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

Ready.


## 8. Imports

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

Imports OK.


## 9. Configuration / Constants

In [3]:
DATA_DIR = Path('.')
CSV_FILE = DATA_DIR / 'data.csv'
RANDOM_STATE = 42

## 10. Dataset Download / Loading

In [4]:
# Dataset is available locally as data.csv
# Alternatively download from Kaggle:
# subprocess.run(['kaggle','datasets','download','-d','mirichoi0218/insurance','-p',str(DATA_DIR),'--unzip'], capture_output=True, text=True)
df = pd.read_csv(CSV_FILE)
print(f'Shape: {df.shape}')
df.head()

Shape: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## 11. Data Validation Checks

In [5]:
print('Columns:', df.columns.tolist())
print(f'\nDuplicates: {df.duplicated().sum()}')
print('\nMissing values:')
print(df.isnull().sum())
print('\nData types:')
print(df.dtypes)
df.describe()

Columns: ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']

Duplicates: 1

Missing values:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Data types:
age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object


,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


## 12. Data Cleaning
1. Check for duplicates and remove if any
2. Verify data types
3. Encode categorical variables for analysis

In [6]:
df = df.drop_duplicates()
print(f'After dedup: {df.shape}')

# Encode categoricals for correlation analysis
df['smoker_flag'] = (df['smoker'] == 'yes').astype(int)
df['sex_flag'] = (df['sex'] == 'male').astype(int)
print('Encoding complete.')

After dedup: (1337, 7)
Encoding complete.


## 13. Exploratory Data Analysis

In [7]:
print('Charges statistics:')
print(df['charges'].describe())
print(f'\nMedian charges: ${df["charges"].median():,.2f}')
print(f'Skewness: {df["charges"].skew():.2f}')
print(f'\nSmokers: {df["smoker"].value_counts().to_dict()}')
print(f'Regions: {df["region"].value_counts().to_dict()}')

Charges statistics:
count     1337.000000
mean     13279.121487
std      12110.359656
min       1121.873900
25%       4746.344000
50%       9386.161300
75%      16657.717450
max      63770.428010
Name: charges, dtype: float64

Median charges: $9,386.16
Skewness: 1.52

Smokers: {'no': 1063, 'yes': 274}
Regions: {'southeast': 364, 'southwest': 325, 'northwest': 324, 'northeast': 324}


## 14. Univariate Analysis

In [8]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].hist(df['charges'], bins=40, color='steelblue', edgecolor='white')
axes[0,0].set_title('Distribution of Charges')
axes[0,0].set_xlabel('Charges ($)')

axes[0,1].hist(df['age'], bins=20, color='coral', edgecolor='white')
axes[0,1].set_title('Age Distribution')

axes[0,2].hist(df['bmi'], bins=30, color='seagreen', edgecolor='white')
axes[0,2].set_title('BMI Distribution')

df['sex'].value_counts().plot(kind='bar', ax=axes[1,0], color=['#457b9d','#e63946'])
axes[1,0].set_title('Sex Distribution')
axes[1,0].tick_params(axis='x', rotation=0)

df['smoker'].value_counts().plot(kind='bar', ax=axes[1,1], color=['#2a9d8f','#e76f51'])
axes[1,1].set_title('Smoker Distribution')
axes[1,1].tick_params(axis='x', rotation=0)

df['children'].value_counts().sort_index().plot(kind='bar', ax=axes[1,2], color='slateblue')
axes[1,2].set_title('Number of Children')
axes[1,2].tick_params(axis='x', rotation=0)

plt.tight_layout(); plt.show()

## 15. Bivariate / Multivariate Analysis

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=df, x='smoker', y='charges', ax=axes[0], palette='Set2')
axes[0].set_title('Charges by Smoking Status')

sns.scatterplot(data=df, x='age', y='charges', hue='smoker', alpha=0.6, ax=axes[1])
axes[1].set_title('Age vs Charges (colored by Smoker)')

sns.scatterplot(data=df, x='bmi', y='charges', hue='smoker', alpha=0.6, ax=axes[2])
axes[2].set_title('BMI vs Charges (colored by Smoker)')

plt.tight_layout(); plt.show()

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='region', y='charges', ax=axes[0], palette='pastel')
axes[0].set_title('Charges by Region')

sns.boxplot(data=df, x='children', y='charges', ax=axes[1], palette='muted')
axes[1].set_title('Charges by Number of Children')

plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
### Correlation Heatmap
Examine linear correlations among numeric features including encoded categoricals.

In [11]:
numeric_cols = ['age','bmi','children','charges','smoker_flag','sex_flag']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlation Matrix', fontsize=14)
plt.tight_layout(); plt.show()

print('\nTop correlations with charges:')
print(corr['charges'].drop('charges').sort_values(ascending=False))


Top correlations with charges:
smoker_flag    0.787234
age            0.298308
bmi            0.198401
children       0.067389
sex_flag       0.058044
Name: charges, dtype: float64


## 17. Statistical Checks / Hypothesis Testing
**Test 1:** Do smokers pay significantly more than non-smokers?
**Test 2:** Does region affect charges? (one-way ANOVA)

In [12]:
# T-test: smokers vs non-smokers
smoker_charges = df[df['smoker']=='yes']['charges']
non_smoker_charges = df[df['smoker']=='no']['charges']
t, p = stats.ttest_ind(smoker_charges, non_smoker_charges)
print(f'Smoker mean:     ${smoker_charges.mean():>10,.2f}')
print(f'Non-smoker mean: ${non_smoker_charges.mean():>10,.2f}')
print(f't={t:.3f}, p={p:.2e}')
print(f'Significant difference: {p < 0.05}\n')

# ANOVA: region
groups = [g['charges'].values for _, g in df.groupby('region')]
f_stat, p_val = stats.f_oneway(*groups)
print(f'ANOVA F={f_stat:.3f}, p={p_val:.4f}')
print(f'Region effect significant: {p_val < 0.05}')

Smoker mean:     $ 32,050.23
Non-smoker mean: $  8,440.66
t=46.645, p=1.41e-282
Significant difference: True

ANOVA F=2.926, p=0.0328
Region effect significant: True


## 18. Visual Analysis
### Smoker × BMI Interaction and Age Bands

In [13]:
df['age_band'] = pd.cut(df['age'], bins=[17,30,45,65], labels=['18-30','31-45','46-64'])
df['bmi_cat'] = pd.cut(df['bmi'], bins=[0,25,30,100], labels=['Normal','Overweight','Obese'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=df, x='age_band', y='charges', hue='smoker', ax=axes[0], palette='Set1', ci=None)
axes[0].set_title('Mean Charges by Age Band & Smoking')
axes[0].set_ylabel('Mean Charges ($)')

sns.barplot(data=df, x='bmi_cat', y='charges', hue='smoker', ax=axes[1], palette='Set1', ci=None)
axes[1].set_title('Mean Charges by BMI Category & Smoking')
axes[1].set_ylabel('Mean Charges ($)')

plt.tight_layout(); plt.show()

In [14]:
# Pairplot for key features
sns.pairplot(df[['age','bmi','charges','smoker']], hue='smoker',
             palette='Set1', diag_kind='kde', height=2.5)
plt.suptitle('Pairplot: Age, BMI, Charges by Smoking Status', y=1.02)
plt.show()

## 19. Key Findings
1. **Smoking is the #1 cost driver** — smokers pay on average 3–4× more than non-smokers.
2. **Age and charges correlate positively** (r ≈ 0.30), especially among smokers.
3. **BMI above 30 + smoking** creates a compounding effect, leading to the highest charges.
4. **Region has minimal effect** — ANOVA shows no statistically significant difference across US regions.
5. **Number of children** has a weak impact on charges compared to smoking and age.

## 20. Limitations
- Small dataset (1,338 records) limits generalisability.
- No longitudinal or time-series information.
- Missing variables: pre-existing conditions, plan type, income level.
- BMI alone is a crude health metric.

## 21. Recommendations / Next Steps
1. Build a regression model to predict charges from available features.
2. Explore interaction terms (smoker × BMI, smoker × age).
3. Segment customers into risk tiers for premium setting.
4. Collect additional features (plan type, chronic conditions) for richer analysis.
5. Compare results with other insurance datasets for validation.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Treating charges as normally distributed | Charges are right-skewed | Apply log transform before parametric tests |
| Ignoring smoker interaction | Smoker × BMI/age interactions are strong | Add interaction terms or stratify |
| Using sex as a predictive feature without context | May introduce bias | Evaluate fairness implications |
| Not removing duplicates | Inflates sample size | Always check `.duplicated()` |
| Applying linear correlation to categorical features | Misleading with nominal variables | Use point-biserial or encoding |

## 23. Mini Challenge / Exercises
1. **Log transform:** Apply `np.log1p` to charges and re-run the correlation heatmap — does smoker correlation increase?
2. **Segmentation:** Create a 'high-cost' flag (charges > 75th percentile) and profile those patients.
3. **Regression:** Fit a simple OLS model. What R² do you get with just `age + bmi + smoker_flag`?
4. **Bootstrap:** Compute bootstrapped 95% CI for the mean difference between smoker and non-smoker charges.
5. **Outlier investigation:** Are there non-smokers with unusually high charges? What might explain them?

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  Smoking status is the single strongest predictor of insurance cost.
TAKEAWAY 2  Age and BMI contribute positively but their effect is amplified for smokers.
TAKEAWAY 3  Region and sex have minimal effect on charges in this dataset.
TAKEAWAY 4  The charges distribution is right-skewed — log transforms improve analysis.
TAKEAWAY 5  Interaction effects (smoker × BMI) are crucial for accurate cost modelling.
```